In [1]:
!pip install PyGithub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.2 MB/s eta 0:00:00


In [2]:
import json
import time
import requests
from github import Github

# **Configurations**

In [ ]:
GITHUB_TOKEN = "PASET_YOUR_TOKEN"
NUM_REPOS   = 10
COMMITS_PER_REPO = 100
OUTPUT_FILE = "commit_dataset.jsonl"

In [4]:
REPOS_TO_SCRAPE = [
    "python/cpython",           # Python itself
    "pallets/flask",            # Flask web framework
    "psf/requests",             # The "requests" HTTP library
    "django/django",            # Django web framework
    "fastapi/fastapi",          # FastAPI
    "tiangolo/sqlmodel",        # SQLModel
    "encode/httpx",             # HTTPX
    "pytest-dev/pytest",        # Pytest
    "huggingface/transformers", # HuggingFace Transformers
    "openai/openai-python",     # OpenAI Python SDK
]

# **Helper Functions**

In [5]:
# Call github's api to get code diff in plain text format

def get_diff_for_commit(repo_full_name, commit_sha, token):

    url = f"https://api.github.com/repos/{repo_full_name}/commits/{commit_sha}"
    headers = {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3.diff",
    }
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        return response.text
    else:
        return None


In [6]:
# Data cleaning - Take only good commits, exclude which has Merge commits, Tiny messages, Huge diffs, Empty diffs

def is_good_commit(commit_message, diff_text):

    # Skip merge commits
    if commit_message.startswith("Merge"):
        return False

    # Skip if message is too short
    if len(commit_message.strip()) < 10:
        return False

    # Skip if message is too long
    if len(commit_message.strip()) > 200:
        return False

    # Skip if diff is empty
    if not diff_text or len(diff_text.strip()) < 10:
        return False

    # Skip if diff is HUGE
    # Large diffs are hard for the model to learn from
    if len(diff_text) > 3000:
        return False

    return True


In [7]:
# Convert a raw diff + commit message into a training example

def format_as_training_example(diff_text, commit_message, repo_name):

    # Clean up the commit message
    # Take only the first line (the "subject" line)
    first_line = commit_message.strip().split("\n")[0].strip()

    # Truncate diff if it's long
    truncated_diff = diff_text[:2000]
    if len(diff_text) > 2000:
        truncated_diff += "\n... [diff truncated for length]"

    instruction = (
        "You are an expert developer. "
        "Given the following git diff, write a concise and descriptive commit message.\n\n"
        f"Git diff:\n```\n{truncated_diff}\n```\n\n"
        "Write a commit message for this change:"
    )

    return {
        "instruction": instruction,
        "response": first_line,
        "repo": repo_name,           # useful for debugging later
        "diff_length": len(diff_text),
    }


# **Collect Github Data**

In [14]:
# Collect commit data - main function

def collect_data():
    print("Connecting to GitHub...")
    g = Github(GITHUB_TOKEN)

    # Quick check: make sure token works
    try:
        user = g.get_user()
        print(f"Logged in as: {user.login}")
    except Exception as e:
        print(f"ERROR: Could not connect. Check your token! Error: {e}")
        return

    all_examples = []

    for repo_name in REPOS_TO_SCRAPE[:NUM_REPOS]:
        print(f"\nScraping: {repo_name}")

        try:
            repo = g.get_repo(repo_name)
        except Exception as e:
            print(f"Could not access repo: {e}. Skipping.")
            continue

        # Get the most recent commits from the main branch
        try:
            commits = repo.get_commits()
        except Exception as e:
            print(f"Could not get commits: {e}. Skipping.")
            continue

        count = 0
        skipped = 0

        for commit in commits:

            # Stop once we have enough commits from this repo
            if count >= COMMITS_PER_REPO:
                break

            # Get commit message
            commit_message = commit.commit.message

            # Get the diff (this is a separate API call)
            diff_text = get_diff_for_commit(repo_name, commit.sha, GITHUB_TOKEN)

            # Check if this commit is good enough to keep
            if not is_good_commit(commit_message, diff_text):
                skipped += 1
                continue

            # Format it as a training example
            example = format_as_training_example(diff_text, commit_message, repo_name)
            all_examples.append(example)
            count += 1

            # GitHub allows 5000 requests/hour with a token
            time.sleep(0.5)

        print(f"Kept: {count} commits | Skipped: {skipped} commits")

    return all_examples


# **Save Commits Data**

In [9]:
# Save as JSONL format.

def save_dataset(examples):
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for example in examples:
            f.write(json.dumps(example, ensure_ascii=False) + "\n")

    print(f"\nSaved {len(examples)} training examples to: {OUTPUT_FILE}")


In [10]:
# Show a summary of what we collected.

def print_stats(examples):
    if not examples:
        print("No examples collected!")
        return

    avg_diff_len = sum(e["diff_length"] for e in examples) / len(examples)
    avg_msg_len  = sum(len(e["response"]) for e in examples) / len(examples)

    print("\n" + "="*50)
    print("  DATASET SUMMARY")
    print("="*50)
    print(f"  Total training examples : {len(examples)}")
    print(f"  Average diff length     : {avg_diff_len:.0f} chars")
    print(f"  Average commit msg len  : {avg_msg_len:.0f} chars")
    print(f"  Output file             : {OUTPUT_FILE}")
    print("="*50)

    print("\n  Sample example (first one):")
    print("-"*50)
    sample = examples[0]
    print(f"  REPO: {sample['repo']}")
    print(f"  RESPONSE (commit msg): {sample['response']}")
    print(f"  INSTRUCTION (first 200 chars):")
    print(f"  {sample['instruction'][:200]}...")

In [12]:
# Print a few examples so you can see what your data looks like.

def preview_sample(examples, n=3):
    print(f"\n  Showing {n} random examples from dataset:")
    print("-"*50)
    import random
    for ex in random.sample(examples, min(n, len(examples))):
        print(f"\n  Repo   : {ex['repo']}")
        print(f"  Commit : {ex['response']}")
        print(f"  Diff   : {ex['instruction'][ex['instruction'].find('```')+3:][:150]}...")
        print()


# **Main Function**

In [ ]:
if __name__ == "__main__":
    print("Starting data collection...")
    print(f"Will scrape {NUM_REPOS} repos, up to {COMMITS_PER_REPO} commits each")
    print(f"Estimated examples: {NUM_REPOS * COMMITS_PER_REPO // 3} (after filtering)\n")

    examples = collect_data()

    if examples:
        print_stats(examples)
        preview_sample(examples)
        save_dataset(examples)
    else:
        print("Something went wrong — no examples collected.")
